## Notebook and DF prep

pandas can’t talk to Postgres by itself. It needs three helpers:  

sqlalchemy builds the connection  
psycopg2-binary is the Postgres driver (the same one dbt installed in its own venv)  
python-dotenv reads the password from .env  

In [1]:
%pip install sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from dotenv import dotenv_values
from sqlalchemy import create_engine, text

In [5]:
# read the database login from .env (in the repo's main folder, one level up)
config = dotenv_values('../.env')
print(config.keys())   # shows the names only, never the password

odict_keys(['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT', 'DB_NAME'])


In [6]:
# dtype=str: every value stays text, exactly as in the file (types get fixed in dbt staging)
shades = pd.read_csv('../data/processed/shades_color.csv', dtype=str)
ulta = pd.read_csv('../data/raw/cleaned_makeup_products.csv', dtype=str)
sephora = pd.read_csv('../data/raw/product_info.csv', dtype=str)
mapping_prices = pd.read_csv('../data/manual/mapping_prices.csv', dtype=str)

In [ ]:
# checking the file sizes
for name, df in [('shades', shades), ('ulta', ulta), ('sephora', sephora), ('mapping_prices', mapping_prices)]:
    print(name, df.shape)

shades (6791, 19)
ulta (1373, 35)
sephora (8494, 27)
mapping_prices (174, 7)


## Connecting to Postgres, with a test


### Starting the engine

In [8]:
# The engine is like a saved address card. It holds who you are and where the database is, but doesn’t connect yet
# the address of the database, built from the .env values
engine = create_engine(
    f"postgresql+psycopg2://{config['DB_USER']}:{config['DB_PASSWORD']}"
    f"@{config['DB_HOST']}:{config['DB_PORT']}/{config['DB_NAME']}"
)

### The test

In [9]:
# This actually connects and asks Postgres two small questions
# connect once and check: who am I, which database, does the raw schema exist?
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_database()")).fetchone())
    print(conn.execute(text(
        "SELECT schema_name FROM information_schema.schemata WHERE schema_name = 'raw'"
    )).fetchone())

('dbt_user', 'foundation_dw')
('raw',)


The first line means you’re logged in as the right user in the right database. The second means the raw schema created earlier in DBeaver exists.

## Loading into raw


The dictionary pairs each table name with its DataFrame, so one loop can load all four instead of writing to_sql four times. Inside to_sql:

- name, engine: the table name, and the connection from the last section  
- schema='raw': put the table into the raw schema  
- if_exists='replace': if the table already exists, delete it and write it fresh. (That way I can rerun the notebook without getting duplicate rows)  
- index=False: don’t write the pandas row numbers (0, 1, 2, …) as an extra column 

In [10]:
# the four DataFrames, with the table name each one gets in Postgres
tables = {
    'shades': shades,
    'ulta': ulta,
    'sephora': sephora,
    'mapping_prices': mapping_prices,
}

for name, df in tables.items():
    df.to_sql(name, engine, schema='raw', if_exists='replace', index=False)
    print(name, 'loaded')

shades loaded
ulta loaded
sephora loaded
mapping_prices loaded


## Checking that the tables arrived (row counts)

In [ ]:
# number of rows in each Postgres table compared with the DataFrame it came from
with engine.connect() as conn:
    for name, df in tables.items():
        rows_db = conn.execute(text(f"SELECT COUNT(*) FROM raw.{name}")).scalar()
        print(name, '| DataFrame:', len(df), '| Postgres:', rows_db, '| match:', len(df) == rows_db)

shades | DataFrame: 6791 | Postgres: 6791 | match: True
ulta | DataFrame: 1373 | Postgres: 1373 | match: True
sephora | DataFrame: 8494 | Postgres: 8494 | match: True
mapping_prices | DataFrame: 174 | Postgres: 174 | match: True


In [12]:
# column types per table: should be only 'text', with the same column count as the DataFrame
types = pd.read_sql(
    """
    SELECT table_name, data_type, COUNT(*) AS n_columns
    FROM information_schema.columns
    WHERE table_schema = 'raw'
    GROUP BY table_name, data_type
    ORDER BY table_name
    """,
    engine,
)
types

,table_name,data_type,n_columns
0,mapping_prices,text,7
1,sephora,text,27
2,shades,text,19
3,ulta,text,35
